# Second Adversarial Audit — Corrected after source-level challenge

**Erratum.** The earlier notebook incorrectly parsed the original LaTeX source `\frac{\sqrt{2/N}}\varepsilon` as multiplication by `\varepsilon`. In TeX the unbraced `\varepsilon` is the denominator of `\frac`. The archived orthonormal-basis matrix was **correct**. The earlier determinant mismatch tested a matrix that was **not printed** in the source. The assertion that the manuscript had a general-coupling matrix error is **withdrawn**.

Source under test: [`cosine-spectrum.md`, lines 161–188](https://github.com/NuckyDucky/The-Super-Zeta-Function/blob/2e16f34e938fee5ccf22e5f6e59988ffbd38d1f7/archive/2026-11-16-super-zeta-v001/sources/foundation/cosine-spectrum.md#L161-L188), [`balanced-charge-operators.md`](https://github.com/NuckyDucky/The-Super-Zeta-Function/blob/main/archive/2026-11-16-super-zeta-v001/research/balanced-charge-operators.md), [`quarter-shift.md`](https://github.com/NuckyDucky/The-Super-Zeta-Function/blob/main/archive/2026-11-18-lean-foundations-v001/notes/quarter-shift.md).

All calculations below use SymPy/mpmath rather than importing code from the repository. Recompute as desired in free Colab.

In [1]:
import sympy as sp
import mpmath as mp
mp.mp.dps = 65
print('sympy',sp.__version__,'mpmath',mp.__version__)

sympy 1.14.0 mpmath 1.3.0


## 1. Correct coupling in the finite matrix

The source expression is

```latex
(B_N)_{j,N+1}=\frac{\sqrt{2/N}}\varepsilon
```

and therefore means $(B_N)_{j,N+1}=\sqrt{2/N}/\varepsilon$. That agrees with the inverse operator $(\mathcal B_\varepsilon(h,k))_1=\ldots+k/\varepsilon$. Its cellwise constant projection has exactly that upper-right coefficient.

In [2]:
I=sp.I
u=sp.symbols('u')

def projected_matrix(N,e):
    assert N % 2 == 0
    B=sp.zeros(N+1)
    for j in range(N):
        for k in range(N):
            B[j,k]=I*sp.sign(j-k)/N
        B[j,N]=sp.sqrt(sp.Rational(2,N))/e
        sj=-1 if j<N//2 else 1
        B[N,j]=-I*sp.sqrt(sp.Rational(2,N))*sj/2
    B[N,N]=1/e
    return B

def claimed_det(N,e):
    return sp.expand(((1+I*u/N)**N+(1-I*u/N)**N)/2-u/e*(1+u*u/N**2)**(N//2))

for N in (2,4,6,8):
    for e in (sp.Integer(2),1+I):
        B=projected_matrix(N,e)
        D=(sp.eye(N+1)-u*B).det(method='domain-ge')
        residual=sp.simplify(D-claimed_det(N,e))
        assert residual==0, (N,e,residual)
        print('PASS:', 'N=',N, 'epsilon=',e)

print('correct source: N=2, epsilon=2, u=1 ->',claimed_det(2,2).subs(u,1))
wrong=projected_matrix(2,sp.Integer(2))
wrong[0,2]=wrong[1,2]=sp.Integer(2)
print('deliberately wrong multiplication interpretation ->',(sp.eye(3)-wrong).det())
assert claimed_det(2,2).subs(u,1)==sp.Rational(1,8)
assert (sp.eye(3)-wrong).det()==-sp.Rational(5,8)

PASS: N= 2 epsilon= 2
PASS: N= 2 epsilon= 1 + I


PASS: N= 4 epsilon= 2


PASS: N= 4 epsilon= 1 + I


PASS: N= 6 epsilon= 2


PASS: N= 6 epsilon= 1 + I
PASS: N= 8 epsilon= 2


PASS: N= 8 epsilon= 1 + I
correct source: N=2, epsilon=2, u=1 -> 1/8
deliberately wrong multiplication interpretation -> -5/8


The orthogonal projection has squared Hilbert–Schmidt error $1/N$ (from the diagonal cells of the integral kernel) with the **correct** upper-right term. The earlier alleged additive error $2|\varepsilon-\varepsilon^{-1}|^2$ applies only to the fabricated multiplication-coupling matrix.

## 2. Quarter displacement: grid versus actual quartet radii

Let $B_n=4^{-n}\binom{2n}{n}$ and $D_n=\pi nB_n^2$. For actual quartet radius $R_n$, write $\delta_n=n(1-R_nB_n^2/2)$. The exact identity is

$$\delta_n=n(1-D_n)-\frac{D_n}{2\pi}(R_n-2\pi n).$$

The limit $\delta_n\to 1/4$ follows from central-binomial asymptotics plus $R_n-2\pi n\to0$. The finite-level crossing cannot be inferred from the grid limit alone.

In [3]:
dc=(1-1/mp.sqrt(3))/2
print('Hurwitz threshold:',mp.nstr(dc,40))

def radius_at_level(n):
    theta=lambda t: mp.acos(-t/mp.sinh(t))
    F=lambda t: theta(t)+mp.cosh(t)*mp.sin(theta(t))
    guess=mp.log(2*mp.pi*n)
    t=mp.findroot(lambda t:F(t)-(n+mp.mpf('0.5'))*mp.pi,
                  (guess,guess+mp.mpf('0.03')))
    return mp.exp(t*mp.coth(t))

for n in [1,2,28,29,100]:
    B=mp.binomial(2*n,n)/mp.power(4,n)
    D=mp.pi*n*B**2
    grid=n*(1-D)
    R=radius_at_level(n)
    actual=n-n*R*B**2/2
    print(f'n={n:3d}', 'actual=',mp.nstr(actual,24),
          'grid=',mp.nstr(grid,24), 'actual-grid=',mp.nstr(actual-grid,16))
    if n==28: assert actual<dc<grid
    if n==29: assert dc<actual<grid

Hurwitz threshold: 0.2113248654051871177454256097490212721762
n=  1 actual= 0.156641498536998754799443 grid= 0.214601836602551690384339 actual-grid= -0.05796033806555294
n=  2 actual= 0.150765754492558466301664 grid= 0.232854132355741303364763 actual-grid= -0.08208837786318284
n= 28 actual= 0.211094352175677825195608 grid= 0.248874079379779015180234 actual-grid= -0.03777972720410119
n= 29 actual= 0.211856113571527458742553 grid= 0.248913228286001919745083 actual-grid= -0.03705711471447446
n=100 actual= 0.232216858672257747365523 grid= 0.249686721219399294661904 actual-grid= -0.01746986254714155


### Why the grid stays above the threshold at every integer

From the binomial kernel formula deposited in `quarter-shift.md`,

$$\log D_n=-\int_0^\infty e^{-nt}\frac{\tanh(t/4)}{t}\,dt.$$

The elementary inequality $\tanh a>1-e^{-a}$ for $a>0$ gives $D_n<4n/(4n+1)$. (For $q=e^{-a}$ their difference is $q(1-q)^2/(1+q^2)>0$.) Meanwhile $D_{n+1}/D_n=1+1/[4n(n+1)]$. Therefore

$$\delta^{\mathrm{grid}}_{n+1}-\delta^{\mathrm{grid}}_n
=1-D_n\left(1+\frac1{4n}\right)>0.$$

Thus the grid shift is strictly increasing, starts at $1-\pi/4>\delta_c=(1-1/\sqrt3)/2$, and tends to $1/4$. The actual quartet radial term produces the observed finite-level crossing, absent on the reference grid.

In [4]:
for n in range(1,201):
    B=mp.binomial(2*n,n)/mp.power(4,n)
    D=mp.pi*n*B**2
    assert D < 4*mp.mpf(n)/(4*n+1)
    assert n*(1-D)>dc
print('PASS: strict grid inequality and no crossing verified for first 200 n')

PASS: strict grid inequality and no crossing verified for first 200 n


## 3. Four actual inverse branches: independent center checks

The archived construction uses $L_*=i\exp(-\sin d)$ with $d=\cos d$, disk $\overline B(-37/2+32i,20)$, and branch sheet $k=10^6$. The inverse maps of $g_L(z)=\exp(L\exp(-Lz))$ are

$$b_j(w)=-\frac1{L_*}\left[\operatorname{Log}\left(\frac{\operatorname{Log}w+2\pi i k}{L_*}\right)+2\pi i j\right],\qquad j=0,1,2,3.$$

The branch selections and their lattice charge assignments are **choices**, but these are real inverse branches of the original dynamics, not a merely metaphorical pair of histories.

In [5]:
d=mp.findroot(lambda z:z-mp.cos(z), mp.mpf('0.7'))
y=mp.exp(-mp.sin(d)); L=1j*y
c=mp.mpc(-mp.mpf(37)/2,32); r=20; k=10**6
q=1/(y*(abs(c)-r)*(2*mp.pi*k-(mp.log(abs(c)+r)+mp.pi)))
print('Analytic uniform derivative bound q:',mp.nstr(q,24))
for j in range(4):
    bj=-(mp.log((mp.log(c)+2*mp.pi*1j*k)/L)+2*mp.pi*1j*j)/L
    margin=r-(abs(bj-c)+r*q)
    assert margin>0
    print('branch',j,'center displacement=',mp.nstr(abs(bj-c),18),
          'invariance margin >=',mp.nstr(margin,18))
assert q<mp.mpf('1')/50000000

Analytic uniform derivative bound q: 1.84021507196367422842709e-8
branch 0 center displacement= 18.5000145544646182 invariance margin >= 1.49998507749236738
branch 1 center displacement= 6.17677441702498081 invariance margin >= 13.8232252149320048
branch 2 center displacement= 6.14657314169770417 invariance margin >= 13.8534264902592814
branch 3 center displacement= 18.4698131034853352 invariance margin >= 1.53018652847165044


## 4. Revised interpretation

**Corrected:** There was **no printed matrix error**. The finite determinant calculation supports the archived source for the tested sizes and couplings.

**Strengthened:** The balanced-sign normalization has an explicit holomorphic-dynamical realization through four certified inverse branches and their trace difference. That is more than an arbitrary unrelated random walk.

**Still a choice:** Selecting this four-branch subsystem, its equal/isotropic weights, its square-lattice charges, and pairing a word of length $2n$ with quartet index $n$ are additional mathematical constructions. The quarter coefficient is controlled by the central-binomial return probability; the actual radii change the finite-level behavior and create the specific Hurwitz crossing.

**Scope:** This notebook checks algebra, high-precision values, and center/analytic derivative bounds. It does not independently rerun the archived 768-bit four-branch interval certificate or the global directed finite-prefix proof.